# SASRec (Self-Attentive Sequential Recommendation) — Algorithm 2

SASRec models the user's sequential interaction history using a Transformer-style self-attention mechanism to predict the next item a user is likely to interact with.

Features: user interaction sequences ordered by timestamp, item ID embeddings, positional embeddings.

## Part 1: Environment Setup

### Part 1.1: Imports & Random Seed

In [1]:
import random
import numpy as np
import torch
import torch.nn as nn

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Random seed set to {SEED}")

Random seed set to 42


### Part 1.2: Device Check (CPU / GPU)

In [2]:

if torch.cuda.is_available():
    DEVICE = torch.device("cuda:0")
    print(f"GPU available: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    DEVICE = torch.device("cpu")
    print("No GPU found — running on CPU.")

print(f"Device: {DEVICE}")


GPU available: NVIDIA GeForce RTX 4090
Memory: 25.3 GB
Device: cuda:0


### Part 1.3: Hyperparameters

In [3]:

# --- Sequence ---
MAX_SEQ_LEN  = 50      # maximum number of past interactions to consider

# --- Model architecture ---
D_MODEL      = 64      # embedding / hidden dimension
NUM_HEADS    = 2       # number of self-attention heads (D_MODEL must be divisible)
NUM_BLOCKS   = 2       # number of Transformer encoder blocks
DROPOUT      = 0.2     # dropout rate applied to embeddings and attention outputs

# --- Training ---
LR           = 1e-3    # Adam learning rate
EPOCHS       = 50      # maximum training epochs
PATIENCE     = 5       # early stopping patience (epochs without HR@10 improvement)
BATCH_SIZE   = 256     # training batch size
NEG_SAMPLES  = 1       # number of negative samples per positive in BPR/CE loss

# --- Evaluation ---
TOP_K        = 10      # cut-off for HR@K and NDCG@K

# --- Reproducibility ---
RANDOM_STATE = SEED

print("Hyperparameters:")
for name, val in [
    ("MAX_SEQ_LEN", MAX_SEQ_LEN), ("D_MODEL", D_MODEL), ("NUM_HEADS", NUM_HEADS),
    ("NUM_BLOCKS", NUM_BLOCKS), ("DROPOUT", DROPOUT), ("LR", LR),
    ("EPOCHS", EPOCHS), ("PATIENCE", PATIENCE), ("BATCH_SIZE", BATCH_SIZE),
    ("NEG_SAMPLES", NEG_SAMPLES), ("TOP_K", TOP_K),
]:
    print(f"  {name:<15} = {val}")


Hyperparameters:
  MAX_SEQ_LEN     = 50
  D_MODEL         = 64
  NUM_HEADS       = 2
  NUM_BLOCKS      = 2
  DROPOUT         = 0.2
  LR              = 0.001
  EPOCHS          = 50
  PATIENCE        = 5
  BATCH_SIZE      = 256
  NEG_SAMPLES     = 1
  TOP_K           = 10


## Part 2: Load Data & Preprocessing

### Part 2.1: Load CSVs

Load the ratings CSV **keeping the `timestamp` column** — SASRec requires interaction order. Movie metadata is loaded for later use at inference time.

In [ ]:
import os
import pandas as pd

DATA_DIR = os.path.join(os.path.abspath(".."), "flaskr", "static", "ml_data")

# Keep timestamp — critical for building ordered sequences
ratings_df = pd.read_csv(os.path.join(DATA_DIR, "ratings.csv"))
movies_df  = pd.read_csv(os.path.join(DATA_DIR, "movie_info.csv"))

print(f"Ratings : {ratings_df.shape}  |  columns: {list(ratings_df.columns)}")
print(f"Movies  : {movies_df.shape}   |  columns: {list(movies_df.columns)}")
print()
print(ratings_df.head(3))

### Part 2.2: ID Encoding

Encode `userId` and `movieId` to contiguous integer indices.  
Item indices start from **1** — index **0** is reserved as the padding token.

In [ ]:
from sklearn.preprocessing import LabelEncoder

user_enc = LabelEncoder()
item_enc = LabelEncoder()

ratings_df["user_idx"] = user_enc.fit_transform(ratings_df["userId"])
ratings_df["item_idx"] = item_enc.fit_transform(ratings_df["movieId"]) + 1  # 0 = padding

NUM_USERS = int(ratings_df["user_idx"].nunique())
NUM_ITEMS = int(ratings_df["item_idx"].nunique()) + 1  # +1 for padding index 0

print(f"Unique users : {NUM_USERS}")
print(f"Unique items : {NUM_ITEMS - 1}  (vocab size incl. pad token: {NUM_ITEMS})")

### Part 2.3: Build Interaction Sequences

Sort each user's ratings by timestamp to produce a **chronologically ordered** item sequence per user.

In [ ]:
# Sort globally by user then timestamp, then group into per-user sequences
ratings_sorted = ratings_df.sort_values(["user_idx", "timestamp"])
user_sequences = (
    ratings_sorted.groupby("user_idx")["item_idx"]
    .apply(list)
    .to_dict()
)

lengths = [len(v) for v in user_sequences.values()]
print(f"Users with sequences : {len(user_sequences)}")
print(f"Sequence length  —  min: {min(lengths)},  max: {max(lengths)},  "
      f"median: {int(np.median(lengths))},  mean: {np.mean(lengths):.1f}")

### Part 2.4: Sequence Construction & Padding

**Leave-one-out split** (standard for SASRec evaluation):
- **Validation**: input = all items except the last → target = last item  
- **Training**: from the remaining history, input[t] → next-item label[t] at every position  

Sequences shorter than `MAX_SEQ_LEN` are **left-padded** with zeros; longer ones are **right-truncated** (keep the most recent interactions).  
Users with fewer than 3 interactions are skipped.

In [ ]:
def pad_or_truncate(seq, max_len, pad_val=0):
    """Left-pad with pad_val, or truncate to keep the most recent max_len items."""
    seq = list(seq)
    if len(seq) >= max_len:
        return seq[-max_len:]
    return [pad_val] * (max_len - len(seq)) + seq

train_seqs     = []   # padded input sequences          shape (N, MAX_SEQ_LEN)
train_pos      = []   # padded positive next-item seqs  shape (N, MAX_SEQ_LEN)
val_seqs       = []   # padded input sequences for val  shape (M, MAX_SEQ_LEN)
val_targets    = []   # single target item per val user  shape (M,)
user_histories = {}   # user_idx -> set of all interacted item indices (for neg. sampling)

for user_idx, seq in user_sequences.items():
    if len(seq) < 3:   # need at least 3 interactions for a meaningful split
        continue

    user_histories[user_idx] = set(seq)

    # --- Validation sample ---
    # Input: everything except the last item; target: last item
    val_seqs.append(pad_or_truncate(seq[:-1], MAX_SEQ_LEN))
    val_targets.append(seq[-1])

    # --- Training sample ---
    # Exclude the last item (reserved for val), then build next-item prediction pairs:
    #   position t: input = history up to t, label = item at t+1
    train_seq = seq[:-1]
    train_seqs.append(pad_or_truncate(train_seq[:-1], MAX_SEQ_LEN))  # input
    train_pos.append( pad_or_truncate(train_seq[1:],  MAX_SEQ_LEN))  # next-item labels

train_seqs  = np.array(train_seqs,  dtype=np.int64)
train_pos   = np.array(train_pos,   dtype=np.int64)
val_seqs    = np.array(val_seqs,    dtype=np.int64)
val_targets = np.array(val_targets, dtype=np.int64)

print(f"Training samples   : {train_seqs.shape}")
print(f"Validation samples : {val_seqs.shape}")
print(f"Users skipped (seq < 3): {len(user_sequences) - len(val_targets)}")

### Part 2.5: Dataset Summary

In [ ]:
# Fraction of non-padding positions (higher = denser sequences)
train_density = (train_pos != 0).sum() / train_pos.size
val_density   = (val_seqs  != 0).sum() / val_seqs.size

print("=" * 45)
print("  Dataset Summary")
print("=" * 45)
print(f"  Users (train/val)  : {len(train_seqs):>7,}")
print(f"  Item vocabulary    : {NUM_ITEMS:>7,}  (incl. pad)")
print(f"  Sequence length    : {MAX_SEQ_LEN:>7}")
print(f"  Train non-pad ratio: {train_density:>7.1%}")
print(f"  Val   non-pad ratio: {val_density:>7.1%}")
print("=" * 45)
print(f"\nSample train input  : {train_seqs[0]}")
print(f"Sample train labels : {train_pos[0]}")
print(f"Sample val target   : {val_targets[0]}")